# 🏥 MediSeg.AI - Google Colaboratory版

医療画像セグメンテーションAIをGoogle Colaboratoryの無料枠で実行

## ✨ 特徴
- ✅ **無料GPU利用**: T4 GPUで高速処理
- ✅ **Web GUIインターフェース**: ブラウザから簡単操作
- ✅ **外部アクセス可能**: ngrokでどこからでもアクセス
- ✅ **Google Drive連携**: データとモデルの永続化
- ✅ **自動セットアップ**: ワンクリックで環境構築
- ✅ **データセット自動ダウンロード**: すぐに使えるデータセット

## 📋 必要なもの
- Googleアカウント
- （オプション）ngrokアカウント（外部アクセス用）


## 📥 ステップ1: リポジトリのクローンとセットアップ

まず、リポジトリをColab環境にクローンします。


In [ ]:
# リポジトリの取得（ZIP でダウンロード - Colab で git 認証エラーを避ける）
# このリポジトリ (shingo257/MediSeg-AI) を取得。フォークした場合は ZIP_URL を書き換えてください。
REPO_DIR = "/content/MediSeg-AI"
ZIP_URL = "https://github.com/shingo257/MediSeg-AI/archive/main.zip"

import os
import urllib.request
import urllib.error
import zipfile

if not os.path.exists(REPO_DIR):
    zip_path = "/tmp/MediSeg-AI-main.zip"
    print("Downloading repository...")
    for branch in ("main", "master"):
        try:
            url = f"https://github.com/shingo257/MediSeg-AI/archive/{branch}.zip"
            urllib.request.urlretrieve(url, zip_path)
            extract_dir = f"MediSeg-AI-{branch}"
            break
        except urllib.error.HTTPError as e:
            if e.code == 404 and branch == "main":
                continue
            raise
    else:
        raise RuntimeError("Could not download repo (tried main and master)")
    print("Extracting...")
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall("/content")
    os.rename(f"/content/{extract_dir}", REPO_DIR)
    os.remove(zip_path)
    print("Repository ready:", REPO_DIR)
else:
    print("Repository already exists")


In [ ]:
# 作業ディレクトリに移動
import os
import sys

REPO_DIR = "/content/MediSeg-AI"
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

print(f"📂 Working directory: {os.getcwd()}")

# セットアップスクリプトの実行
print("\n🔧 Running setup script...")
!python colab_setup.py


## 💾 ステップ2: Google Driveのマウント（オプション）

モデルやデータセットを永続化する場合に推奨
- セッションが切れてもデータが残る
- モデルの保存と読み込みが可能


In [ ]:
# Google Driveのマウント
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('✅ Google Drive mounted at /content/drive')
    
    # データ保存用ディレクトリの作成
    import os
    drive_data_dir = '/content/drive/MyDrive/MediSeg-AI'
    os.makedirs(drive_data_dir, exist_ok=True)
    os.makedirs(f'{drive_data_dir}/models', exist_ok=True)
    os.makedirs(f'{drive_data_dir}/data', exist_ok=True)
    print(f'✅ Created directories in: {drive_data_dir}')
except Exception as e:
    print(f'⚠️  Google Drive mount failed: {e}')
    print('   Continuing without Drive (data will be lost when session ends)')


## 🌐 ステップ3: ngrok認証トークンの設定（オプション）

外部からアクセスする場合に必要
- 無料アカウント作成: https://dashboard.ngrok.com/signup
- 認証トークン取得: https://dashboard.ngrok.com/get-started/your-authtoken
- トークンがない場合: Colab内のプレビューリンクのみ使用可能


In [ ]:
# ngrokトークンを設定（オプション）
# ⚠️ ここにngrokトークンを貼り付けてください
NGROK_TOKEN = ""  # 例: "2abc123def456ghi789jkl012mno345pq"

if NGROK_TOKEN:
    print("✅ ngrok token configured")
else:
    print("⚠️  No ngrok token - using local access only")
    print("   Get free token at: https://dashboard.ngrok.com/get-started/your-authtoken")


## 🚀 ステップ4: GPU確認とWeb GUIの起動

まずGPUが利用可能か確認してから、Web GUIを起動します。


In [ ]:
# GPUの確認
import torch
print("Checking GPU availability...")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU available: {gpu_name} ({gpu_memory:.1f} GB)")
else:
    print("No GPU - using CPU. Enable GPU: Runtime > Change runtime type > GPU")

print("\n" + "="*60)

# Web GUIの起動（このセルを実行し続けることで接続を維持）
try:
    _token = NGROK_TOKEN
except NameError:
    _token = ""
from colab_launcher import launch_web_gui
launch_web_gui(ngrok_token=_token if _token else None)


## 学習（訓練）を実行する（オプション）

データ不要のデモ、または合成データで train-medical を実行できます。**データセットのダウンロード可否や詳しい手順**はリポジトリの [docs/COLAB_TRAINING_GUIDE.md](https://github.com/shingo257/MediSeg-AI/blob/main/docs/COLAB_TRAINING_GUIDE.md) を参照してください。

In [ ]:
# デモ（データ不要・数秒で完了）
!python main.py demo

# 学習を実行する場合: 先に合成データを生成してから train-medical
# from mediseg_ai.data_loaders.real_datasets import RealDatasetDownloader
# d = RealDatasetDownloader(data_dir="/content/MediSeg-AI/data")
# d.download_dataset("medical_small")
# !python main.py train-medical --data-dir /content/MediSeg-AI/data/medical_small --save-dir /content/MediSeg-AI/results

## 📖 使い方

### 基本的な使い方
1. 上記のセルを**順番に実行**してください
2. Web GUIのURLが表示されます
3. ブラウザでアクセスして使用
4. **重要**: 最後のセル（Web GUI起動）を**実行し続ける**ことで接続を維持

### データセットのダウンロード（オプション）
```python
# データセットをダウンロードする場合
from mediseg_ai.data_loaders.download_datasets import DatasetDownloader

downloader = DatasetDownloader(data_dir='/content/MediSeg-AI/data')
downloader.list_available_datasets()  # 利用可能なデータセット一覧
downloader.download_dataset('medical_small')  # データセットをダウンロード
```

### モデルの保存と読み込み
```python
# Google Driveに保存
import shutil
shutil.copy('/content/MediSeg-AI/models/model.pth', 
            '/content/drive/MyDrive/MediSeg-AI/models/')

# Google Driveから読み込み
shutil.copy('/content/drive/MyDrive/MediSeg-AI/models/model.pth',
            '/content/MediSeg-AI/models/')
```

## ⚠️ 注意事項

### セッション管理
- **Colab無料枠**: セッションは約12時間でタイムアウト
- **データの保存**: 重要なデータはGoogle Driveに保存
- **再接続**: セッションが切れた場合は、最初から再実行

### リソース制限
- **GPU使用時間**: 長時間の使用は制限される場合があります
- **メモリ**: 約12.7GB RAM
- **ディスク**: 一時的なストレージ（セッション終了で削除）

## 🔧 トラブルシューティング

### GPUが利用できない
1. **ランタイム > ランタイムのタイプを変更 > GPU** を選択
2. ランタイムを再起動
3. GPU割り当ての待機（無料枠では待機時間がある場合があります）

### メモリ不足エラー
```python
# 1. 不要な変数を削除
del large_variable
torch.cuda.empty_cache()

# 2. ランタイムを再起動
# ランタイム > ランタイムの再起動

# 3. バッチサイズを小さくする（コード内で設定）
```

### ngrok接続エラー
- ngrokトークンが正しく設定されているか確認
- 無料アカウントの制限を確認（同時接続数1など）
- ポート番号を変更: `launch_web_gui(port=8081)`

### セットアップエラー
```python
# 依存関係を個別にインストール
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu118
!pip install opencv-python matplotlib seaborn pillow flask pyngrok
```

## 📚 詳細情報

- **詳細ガイド**: [COLAB_GUIDE.md](https://github.com/shingo257/MediSeg-AI/blob/main/COLAB_GUIDE.md)
- **使用方法**: [USAGE.md](https://github.com/shingo257/MediSeg-AI/blob/main/USAGE.md)
- **GitHub Issues**: 問題が解決しない場合はIssueを作成してください


## 🔍 ステップ5: システム情報とデータセット確認（オプション）

環境の状態を確認し、データセットをダウンロードできます。


In [ ]:
# システム情報の表示
import torch
import sys
import os
from pathlib import Path

print("="*60)
print("📊 System Information")
print("="*60)

# Python情報
print(f"Python: {sys.version.split()[0]}")

# GPU情報
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f} GB")
    print(f"CUDA Version: {torch.version.cuda}")
else:
    print("GPU: Not available (using CPU)")

# ディスク使用量
try:
    import shutil
    total, used, free = shutil.disk_usage('/content')
    print(f"\n💾 Disk Usage:")
    print(f"   Total: {total / (1024**3):.1f} GB")
    print(f"   Used: {used / (1024**3):.1f} GB")
    print(f"   Free: {free / (1024**3):.1f} GB")
except:
    pass

# 作業ディレクトリ
print(f"\n📂 Working Directory: {os.getcwd()}")

# 利用可能なデータセット
try:
    from mediseg_ai.data_loaders.real_datasets import RealDatasetDownloader
    downloader = RealDatasetDownloader(data_dir='/content/MediSeg-AI/data')
    print(f"\n📦 Available Datasets:")
    for dataset_id, info in downloader.datasets.items():
        status = "✅" if downloader.get_dataset_path(dataset_id) else "⬇️"
        print(f"   {status} {info['name']} ({dataset_id})")
except Exception as e:
    print(f"\n⚠️  Could not load dataset information: {e}")

print("="*60)


### データセットのダウンロード

利用可能なデータセットをダウンロードできます。


In [ ]:
# データセットのダウンロード
# ⚠️ データセットによっては時間がかかります

from mediseg_ai.data_loaders.real_datasets import RealDatasetDownloader

# データセットダウンローダーの初期化
downloader = RealDatasetDownloader(data_dir='/content/MediSeg-AI/data')

# 利用可能なデータセット一覧を表示
print("📦 Available Datasets:")
print("-" * 60)
for dataset_id, info in downloader.datasets.items():
    print(f"\n{info['name']} ({dataset_id})")
    print(f"  Description: {info['description']}")
    print(f"  Size: {info['size']}")
    print(f"  Format: {info['format']}")

print("\n" + "="*60)
print("💡 To download a dataset, use:")
print("   downloader.download_dataset('dataset_id')")
print("="*60)

# 例: 小さな医療データセットをダウンロード（コメントアウトを外して実行）
# downloader.download_dataset('medical_small')


### メモリ管理とクリーンアップ

Colabのメモリ制限内で効率的に動作するためのヘルパー関数


In [ ]:
# メモリ管理ヘルパー関数

def clear_memory():
    """不要な変数を削除してメモリを解放"""
    import gc
    import torch
    
    # Pythonのガベージコレクション
    gc.collect()
    
    # PyTorchのキャッシュクリア
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    
    print("✅ Memory cleared")

def show_memory_usage():
    """現在のメモリ使用量を表示"""
    import torch
    
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / (1024**3)
        reserved = torch.cuda.memory_reserved() / (1024**3)
        total = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        
        print("📊 GPU Memory Usage:")
        print(f"   Allocated: {allocated:.2f} GB")
        print(f"   Reserved: {reserved:.2f} GB")
        print(f"   Total: {total:.2f} GB")
        print(f"   Free: {total - reserved:.2f} GB")
    else:
        print("⚠️  GPU not available")

# 使用例:
# clear_memory()  # メモリをクリア
# show_memory_usage()  # メモリ使用量を表示

print("✅ Memory management functions loaded")
print("   Use clear_memory() to free memory")
print("   Use show_memory_usage() to check GPU memory")


## 🎯 クイックスタート（上級者向け）

すべてのセットアップを一度に実行する場合


## 🎯 クイックスタート（上級者向け）

すべてのセットアップを一度に実行する場合


## 🧪 テストと検証

環境が正しくセットアップされているか確認します。


In [ ]:
# 環境テスト: すべてのコンポーネントが正しく動作するか確認

print("🧪 Testing MediSeg.AI Environment...")
print("="*60)

# 1. PyTorchテスト
try:
    import torch
    print("✅ PyTorch imported")
    if torch.cuda.is_available():
        print(f"✅ CUDA available: {torch.cuda.get_device_name(0)}")
    else:
        print("⚠️  CUDA not available")
except Exception as e:
    print(f"❌ PyTorch error: {e}")

# 2. モデルインポートテスト
try:
    from mediseg_ai.models.unet import UNet
    from mediseg_ai.models.deeplabv3plus import create_deeplabv3plus
    print("✅ Models imported")
except Exception as e:
    print(f"❌ Model import error: {e}")

# 3. Web GUIモジュールインポートテスト
try:
    from backend.gui.web import web_gui
    print("Web GUI module imported")
except Exception as e:
    print(f"Web GUI import error: {e}")

# 4. モデル初期化テスト
try:
    from backend.gui.web.services import setup_models
    from backend.gui.web import state as app_state
    if setup_models():
        models = list(app_state.models.keys())
        print(f"Models initialized: {', '.join(models)}")
    else:
        print("Model setup failed")
except Exception as e:
    print(f"Model setup error: {e}")

# 5. データローダーテスト
try:
    from mediseg_ai.data_loaders.real_datasets import RealDatasetDownloader
    downloader = RealDatasetDownloader(data_dir='/content/MediSeg-AI/data')
    print(f"✅ Dataset downloader initialized")
except Exception as e:
    print(f"⚠️  Dataset downloader error: {e}")

print("="*60)
print("✅ Environment test complete!")


## 💡 便利なヘルパー関数

よく使う操作を簡単にする関数集


In [ ]:
# 便利なヘルパー関数

def save_to_drive(file_path, drive_path='/content/drive/MyDrive/MediSeg-AI'):
    """ファイルをGoogle Driveに保存"""
    import shutil
    import os
    
    if not os.path.exists('/content/drive'):
        print("⚠️  Google Drive not mounted")
        return False
    
    os.makedirs(drive_path, exist_ok=True)
    dest = os.path.join(drive_path, os.path.basename(file_path))
    shutil.copy(file_path, dest)
    print(f"✅ Saved to: {dest}")
    return True

def load_from_drive(file_name, drive_path='/content/drive/MyDrive/MediSeg-AI', dest_path=None):
    """Google Driveからファイルを読み込み"""
    import shutil
    import os
    
    if not os.path.exists('/content/drive'):
        print("⚠️  Google Drive not mounted")
        return False
    
    source = os.path.join(drive_path, file_name)
    if not os.path.exists(source):
        print(f"❌ File not found: {source}")
        return False
    
    if dest_path is None:
        dest_path = f'/content/MediSeg-AI/{file_name}'
    
    shutil.copy(source, dest_path)
    print(f"✅ Loaded from Drive: {dest_path}")
    return True

def check_session_time():
    """セッションの残り時間を確認（概算）"""
    import time
    # Colabのセッションタイムアウトは約12時間
    # 正確な時間は取得できないため、開始時刻を記録する必要がある
    print("💡 Colab free tier sessions typically last ~12 hours")
    print("   Save your work regularly to Google Drive!")

def list_files(directory, pattern="*"):
    """ディレクトリ内のファイル一覧を表示"""
    from pathlib import Path
    import os
    
    if not os.path.exists(directory):
        print(f"❌ Directory not found: {directory}")
        return
    
    path = Path(directory)
    files = list(path.glob(pattern))
    
    print(f"📁 Files in {directory}:")
    for f in sorted(files)[:20]:  # 最初の20個のみ表示
        size = f.stat().st_size / (1024**2) if f.is_file() else 0
        print(f"   {'📄' if f.is_file() else '📁'} {f.name} ({size:.2f} MB)" if f.is_file() else f"   📁 {f.name}/")
    
    if len(files) > 20:
        print(f"   ... and {len(files) - 20} more files")

print("✅ Helper functions loaded")
print("   - save_to_drive(file_path): Save file to Google Drive")
print("   - load_from_drive(file_name): Load file from Google Drive")
print("   - check_session_time(): Check session information")
print("   - list_files(directory): List files in directory")


## 📝 使用例とサンプルコード

実際の使用例を示します。


In [ ]:
# 使用例: 画像のセグメンテーション処理

# このセルは参考用です。実際の処理はWeb GUIから行います。

def example_segmentation():
    """セグメンテーション処理の例"""
    import cv2
    import numpy as np
    import torch
    from backend.gui.web.services import setup_models

    # モデルのセットアップ（Web GUIと同じ初期化）
    setup_models()
    
    # 画像の読み込み（例）
    # image_path = '/content/MediSeg-AI/data/medical_small/images/sample_000.png'
    # image = cv2.imread(image_path)
    # image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    # セグメンテーション処理
    # result, error = model_service.process_image(image, 'U-Net', threshold=0.5)
    
    # if result:
    #     print("✅ Segmentation successful")
    #     print(f"   Prediction shape: {result['prediction'].shape}")
    #     print(f"   Binary mask shape: {result['binary_mask'].shape}")
    # else:
    #     print(f"❌ Error: {error}")
    
    print("💡 This is an example function")
    print("   Actual processing is done through the Web GUI")
    print("   Upload images via the Web GUI interface")

# 実行（コメントアウトを外して実行）
# example_segmentation()


In [ ]:
# クイックスタート: すべてのセットアップを一度に実行
# ⚠️ 上記のセルを個別に実行することを推奨します

import os
import sys

# 1. リポジトリのクローン（既に実行済みの場合はスキップ）
REPO_URL = "https://github.com/shingo257/MediSeg-AI.git"
REPO_DIR = "/content/MediSeg-AI"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

# 2. セットアップ
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
!python colab_setup.py

# 3. GPU確認
import torch
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  No GPU - enable in Runtime settings")

# 4. Web GUI起動
NGROK_TOKEN = ""  # オプション: ngrokトークンを設定
from colab_launcher import launch_web_gui
launch_web_gui(ngrok_token=NGROK_TOKEN if NGROK_TOKEN else None)
